# Install and Setup

In [ ]:
%pip install -q torch torchaudio datasets jiwer soundfile


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torchaudio
import torch.nn as nn
from datasets import load_dataset

In [ ]:
import subprocess
subprocess.run(["wget", "-q", "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"])
subprocess.run(["tar", "-xjf", "LJSpeech-1.1.tar.bz2"])
print("done")

In [ ]:
import os
print(os.listdir("LJSpeech-1.1"))

['README', 'wavs', 'metadata.csv']


In [ ]:
import pandas as pd
df = pd.read_csv("LJSpeech-1.1/metadata.csv", sep="|", header=None, names=["id", "transcript", "normalized"])
df

,id,transcript,normalized
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...
3,LJ001-0004,"produced the block books, which were the immed...","produced the block books, which were the immed..."
4,LJ001-0005,the invention of movable metal letters in the ...,the invention of movable metal letters in the ...
...,...,...,...
13095,LJ050-0274,made certain recommendations which it believes...,made certain recommendations which it believes...
13096,LJ050-0275,materially improve upon the procedures in effe...,materially improve upon the procedures in effe...
13097,LJ050-0276,"As has been pointed out, the Commission has no...","As has been pointed out, the Commission has no..."
13098,LJ050-0277,with the active cooperation of the responsible...,with the active cooperation of the responsible...


In [ ]:
!apt-get install -y libsndfile1
!pip install soundfile --upgrade

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libsndfile1

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
import torchaudio
waveform, sample_rate = torchaudio.load("LJSpeech-1.1/wavs/LJ001-0001.wav")
print("Shape:", waveform.shape)
print("Sample rate:", sample_rate)
print("Duration:", waveform.shape[1] / sample_rate, "seconds")

Shape: torch.Size([1, 212893])
Sample rate: 22050
Duration: 9.65501133786848 seconds


# Convert the raw audio into mel spectrogram

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=22050,
    n_fft=1024,
    hop_length=256,
    n_mels=80
)

mel = mel_transform(waveform)
print("Mel spectrogram shape:", mel.shape)

Mel spectrogram shape: torch.Size([1, 80, 832])


# we went from a 1D list of 212,893 numbers to a 2D grid of 80×832. That 2D grid is the "picture" the CNN will read

 # spectrogram is basically saying — every 11.6ms, what frequencies are present in the audio? 832 times across 9.65 seconds.

# Bulid the vocab of chracters are model can predict

In [ ]:
chars = "abcdefghijklmnopqrstuvwxyz '"
char2idx = {c: i+1 for i, c in enumerate(chars)}
char2idx['<blank>'] = 0
idx2char = {i: c for c, i in char2idx.items()}

print(char2idx)
print("Vocab size:", len(char2idx))

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, ' ': 27, "'": 28, '<blank>': 0}
Vocab size: 29


# Funtion to convert the transcript to numbers

In [ ]:
def text_to_indices(text):
    text = text.lower()
    return [char2idx[c] for c in text if c in char2idx]

# test it
sample_text = df['normalized'][0]
print("Text:", sample_text)
print("Indices:", text_to_indices(sample_text))

Text: Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition
Indices: [16, 18, 9, 14, 20, 9, 14, 7, 27, 9, 14, 27, 20, 8, 5, 27, 15, 14, 12, 25, 27, 19, 5, 14, 19, 5, 27, 23, 9, 20, 8, 27, 23, 8, 9, 3, 8, 27, 23, 5, 27, 1, 18, 5, 27, 1, 20, 27, 16, 18, 5, 19, 5, 14, 20, 27, 3, 15, 14, 3, 5, 18, 14, 5, 4, 27, 4, 9, 6, 6, 5, 18, 19, 27, 6, 18, 15, 13, 27, 13, 15, 19, 20, 27, 9, 6, 27, 14, 15, 20, 27, 6, 18, 15, 13, 27, 1, 12, 12, 27, 20, 8, 5, 27, 1, 18, 20, 19, 27, 1, 14, 4, 27, 3, 18, 1, 6, 20, 19, 27, 18, 5, 16, 18, 5, 19, 5, 14, 20, 5, 4, 27, 9, 14, 27, 20, 8, 5, 27, 5, 24, 8, 9, 2, 9, 20, 9, 15, 14]


# Build the dataset class

In [ ]:
from torch.utils.data import Dataset

class LJSpeechDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=22050,
            n_fft=1024,
            hop_length=256,
            n_mels=80
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav_path = f"LJSpeech-1.1/wavs/{row['id']}.wav"
        waveform, _ = torchaudio.load(wav_path)
        mel = self.mel_transform(waveform)
        mel = mel.squeeze(0)  # remove channel dim, shape: [80, T]
        label = torch.tensor(text_to_indices(row['normalized']))
        return mel, label

# test it
dataset = LJSpeechDataset(df)
mel, label = dataset[0]
print("Mel shape:", mel.shape)
print("Label:", label)

Mel shape: torch.Size([80, 832])
Label: tensor([16, 18,  9, 14, 20,  9, 14,  7, 27,  9, 14, 27, 20,  8,  5, 27, 15, 14,
        12, 25, 27, 19,  5, 14, 19,  5, 27, 23,  9, 20,  8, 27, 23,  8,  9,  3,
         8, 27, 23,  5, 27,  1, 18,  5, 27,  1, 20, 27, 16, 18,  5, 19,  5, 14,
        20, 27,  3, 15, 14,  3,  5, 18, 14,  5,  4, 27,  4,  9,  6,  6,  5, 18,
        19, 27,  6, 18, 15, 13, 27, 13, 15, 19, 20, 27,  9,  6, 27, 14, 15, 20,
        27,  6, 18, 15, 13, 27,  1, 12, 12, 27, 20,  8,  5, 27,  1, 18, 20, 19,
        27,  1, 14,  4, 27,  3, 18,  1,  6, 20, 19, 27, 18,  5, 16, 18,  5, 19,
         5, 14, 20,  5,  4, 27,  9, 14, 27, 20,  8,  5, 27,  5, 24,  8,  9,  2,
         9, 20,  9, 15, 14])


# every time PyTorch asks for sample 0, it gets back: A grid [80, 832] — the spectrogram A tensor of indices — the transcript as numbers

# We need to make sure that audio have same width

In [ ]:
def collate_fn(batch):
    mels, labels = zip(*batch)

    # pad mels to same length
    mels = [m.transpose(0, 1) for m in mels]  # [T, 80]
    mels = pad_sequence(mels, batch_first=True)  # [B, T, 80]
    mels = mels.transpose(1, 2)  # [B, 80, T]

    input_lengths = torch.tensor([mels.shape[2]] * len(batch))

    labels_padded = pad_sequence(labels, batch_first=True)
    label_lengths = torch.tensor([len(l) for l in labels])

    return mels, labels_padded, input_lengths, label_lengths

loader = DataLoader(dataset, batch_size=8, collate_fn=collate_fn)
mels, labels, input_lengths, label_lengths = next(iter(loader))
print("Mels shape:", mels.shape)
print("Labels shape:", labels.shape)
print("Input lengths:", input_lengths)
print("Label lengths:", label_lengths)

Mels shape: torch.Size([8, 80, 833])
Labels shape: torch.Size([8, 154])
Input lengths: tensor([833, 833, 833, 833, 833, 833, 833, 833])
Label lengths: tensor([149,  29, 154,  87, 142,  72, 109,  24])


# Model Arch

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CTCModel(nn.Module):
    def __init__(self, n_mels=80, num_classes=29):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
        )

        self.fc = nn.Linear(128 * n_mels, num_classes)

    def forward(self, x):
        # x: [B, 80, T]
        x = x.unsqueeze(1)          # [B, 1, 80, T]
        x = self.cnn(x)             # [B, 128, 80, T]
        x = x.permute(3, 0, 1, 2)  # [T, B, 128, 80]
        x = x.flatten(2)            # [T, B, 128*80]
        x = self.fc(x)              # [T, B, 29]
        x = F.log_softmax(x, dim=2)
        return x

model = CTCModel()
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

# Training Loop

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CTCModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

def train(model, loader, optimizer, ctc_loss, device):
    model.train()
    total_loss = 0

    for batch_idx, (mels, labels, input_lengths, label_lengths) in enumerate(loader):
        mels = mels.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(mels)           # [T, B, 29]
        loss = ctc_loss(outputs, labels, input_lengths, label_lengths)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")

    return total_loss / len(loader)

# run training
for epoch in range(10):
    loss = train(model, loader, optimizer, ctc_loss, device)
    print(f"Epoch {epoch+1}, Avg Loss: {loss:.4f}")

# Decoder

In [ ]:
def greedy_decode(output, idx2char):
    # output: [T, 29] — probabilities for one sample
    indices = torch.argmax(output, dim=1)  # pick highest prob char at each frame

    # collapse repeats and remove blanks
    result = []
    prev = None
    for idx in indices:
        idx = idx.item()
        if idx != prev:          # not a repeat
            if idx != 0:         # not a blank
                result.append(idx2char[idx])
        prev = idx

    return ''.join(result)